# NER Performance Evaluation Pipeline

In [1]:
'''
Function: ner_zero_shot
Description:
 - Performs zero-shot Named Entity Recognition (NER) using the GPT-4o model.
 - Takes instructions, input text, model name, and temperature value as parameters.
 - Sends the instructions and the text to the OpenAI chat API.
 - Receives and cleans the model's response.
 - Attempts to parse the response as JSON.
 - Returns the parsed output if valid, otherwise returns None.
'''

from openai import OpenAI
import json

def ner_zero_shot(instruction, paragraph, model, temp):
    client = OpenAI()
    user_query = 'TEXT: {paragraph}'
    
    response = client.chat.completions.create(
      model = model,
      temperature = temp,
      messages = [
        {'role': 'system', 'content': instruction},
        {'role': 'user', 'content': user_query.format(paragraph=paragraph)}
      ]
    )

    content = response.choices[0].message.content
    cleaned_content = content.strip('```python\n').strip('```')
    
    try:
        output = json.loads(cleaned_content)
        return output
    except json.JSONDecodeError as e:
        return None
    

In [17]:
import os

def set_download_path(dir_path, file_name):
    # set to current working directory if not provided
    if dir_path is None:
        dir_path = os.getcwd()

    os.makedirs(dir_path, exist_ok=True)  # ensure directory exists
    download_path = os.path.join(dir_path, file_name)
    
    return download_path

In [11]:
'''
Function: evaluate_distinct_entities
Description:
 - Evaluates Named Entity Recognition (NER) performance at both paragraph and document levels.
 - Takes labels, paragraphs, predicted and gold standard entities from corresponding paragraphs, 
   and download location as parameters.
 - Calculates precision, recall, and F1-score for each label in each paragraph.
 - Aggregates entity sets across paragraphs to compute overall metrics per label.
 - Saves detailed outputs including:
   - A .txt file logging paragraph-level predictions and gold terms.
   - An Excel file for paragraph-level performance.
   - An Excel file for overall (document-level) performance.
'''

import pandas as pd

# NEED PYTHON 3.9 OR MORE FOR TYPE DESCRIPTION
# def evaluate_distinct_entity(
#     label: list[str],
#     paragraph: list[str],
#     gold_entity: list[dict[str, list[str]]],
#     pred_entity: list[dict[str, list[str]]],
#     dir_path: str
# ) -> None:
    
def evaluate_distinct_entity(label: list, paragraph: list, gold_entity: list, pred_entity: list, dir_path: str=None) -> None:
    
    # paragraph-level performance calculation 
    # variable declaration
    all_gold_ent = {
        'chemical': set(),
        'material': set(),
        'structure': set(),
        'property': set(),
        'application': set(),
        'process': set(),
        'equipment': set(),
        'measurement': set(),
        'abbreviation': set()
    }
    all_pred_ent = {
        'chemical': set(),
        'material': set(),
        'structure': set(),
        'property': set(),
        'application': set(),
        'process': set(),
        'equipment': set(),
        'measurement': set(),
        'abbreviation': set()
    }
    df_score_para = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_para = []
    eval_value_para = []
    
    # retrieving paragraph, gold standard terms, and predicted terms from zipped list
    for para, gold_ent, pred_ent in zip(paragraph, gold_entity, pred_entity):
        para_index = paragraph.index(para)
        
        # writing paragraph number, paragraph, gold standard terms, and predicted terms in a text file
        file_name = 'paragraph-with-all-entity.txt'
        download_path = set_download_path(dir_path, file_name)
        with open(download_path, 'a', encoding='utf-8') as file:
            file.write(f'PARAGRAPH NUMBER: {para_index}\n')
            file.write(f'PARAGRAPH: {para}\n')
            file.write(f'GOLD ENTITY: {gold_ent}\n')
            file.write(f'PRED ENTITY: {pred_ent}\n')
            file.write('=======================================================\n')
        
        if dir_path:
            print(f'Downloaded {file_name} => {dir_path}')
        else:
            print(f'Downloaded {file_name} => root directory')
        
        # retrieving labels from list
        for l in label:
            
            # creating entity set with unique entities
            unique_gold_ent = set(gold_ent[l])
            unique_pred_ent = set(pred_ent[l])
            
            # storing entities (label-wise) for document-level calculation
            all_gold_ent[l].update(unique_gold_ent)
            all_pred_ent[l].update(unique_pred_ent)
            
            # calculate confusion matrix
            tp = unique_gold_ent & unique_pred_ent
            fp = unique_pred_ent - unique_gold_ent
            fn = unique_gold_ent - unique_pred_ent
            
            # calculate precision, recall and f1-score for each paragraph
            precision = len(tp) / (len(tp) + len(fp)) if unique_pred_ent else 0
            recall = len(tp) / (len(tp) + len(fn)) if unique_gold_ent else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            # storing performance for each paragraph in list
            eval_metric_para.extend([f'{para_index} {l.upper()} Precision', 
                                     f'{para_index} {l.upper()} Recall', 
                                     f'{para_index} {l.upper()} F1'])

            eval_value_para.extend([f'{precision:.2f}',
                                    f'{recall:.2f}', 
                                    f'{f1:.2f}'])
    
    # creating dataframe from list
    df_score_para['metric'] = eval_metric_para
    df_score_para['value'] = eval_value_para
    
    # downloading paragraph-level performace in a spreadsheet
    file_name = 'score-para-DIS-ENT.xlsx'
    download_path = set_download_path(dir_path, file_name)
    df_score_para.to_excel(download_path, index=False)
    
    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => root directory')
    
    # document-level performance calculation  
    # variable declaration
    df_score_doc = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_doc = []
    eval_value_doc = []
    
    # retrieving label from list
    for l in label:

        # calculate confusion matrix
        tp = all_gold_ent[l] & all_pred_ent[l]
        fp = all_pred_ent[l] - all_gold_ent[l]
        fn = all_gold_ent[l] - all_pred_ent[l]
        
        # calculate precision, recall and f1-score for entire document
        precision = len(tp) / (len(tp) + len(fp)) if all_pred_ent[l] else 0
        recall = len(tp) / (len(tp) + len(fn)) if all_gold_ent[l] else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f'{l}:\t{f1:.2f}(f) | {precision:.2f}(p) | {recall:.2f}(r)')
        
        # storing performance for the entire document in list
        eval_metric_doc.extend([f'{l.upper()} (Overall) Precision', 
                                f'{l.upper()} (Overall) Recall', 
                                f'{l.upper()} (Overall) F1'])
        
        eval_value_doc.extend([f'{precision:.2f}',
                               f'{recall:.2f}', 
                               f'{f1:.2f}'])

    # creating dataframe from list
    df_score_doc['metric'] = eval_metric_doc
    df_score_doc['value'] = eval_value_doc

    # downloading document-level performace in a spreadsheet
    file_name = 'score-doc-DIS-ENT.xlsx'
    download_path = set_download_path(dir_path, file_name)
    df_score_doc.to_excel(download_path, index=False)
    
    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => root directory')
    

In [12]:
'''
Function: evaluate_all_entities
Description:
 - Evaluates Named Entity Recognition (NER) performance at both paragraph and document levels.
 - Accepts predicted terms, gold standard terms, and corresponding paragraphs for comparison.
 - Calculates precision, recall, and F1-score for each label in each paragraph.
 - Aggregates entity sets across paragraphs to compute overall metrics per label.
 - Saves detailed outputs including:
   - A .txt file logging paragraph-level predictions and gold terms.
   - An Excel file for paragraph-level performance.
   - An Excel file for overall (document-level) performance.
'''

# from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from collections import Counter

def evaluate_all_entity(label: list, paragraph: list, gold_entity: list, pred_entity: list, dir_path: str=None) -> None:
    
    # paragraph-level performance calculation 
    # variable declaration
    all_gold_ent = {
        'chemical': Counter(),
        'material': Counter(),
        'structure': Counter(),
        'property': Counter(),
        'application': Counter(),
        'process': Counter(),
        'equipment': Counter(),
        'measurement': Counter(),
        'abbreviation': Counter()
    }
    all_pred_ent = {
        'chemical': Counter(),
        'material': Counter(),
        'structure': Counter(),
        'property': Counter(),
        'application': Counter(),
        'process': Counter(),
        'equipment': Counter(),
        'measurement': Counter(),
        'abbreviation': Counter()
    }
    df_score_para = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_para = []
    eval_value_para = []
    
    # retrieving paragraph, gold standard terms, and predicted terms from zipped list
    for para, gold_ent, pred_ent in zip(paragraph, gold_entity, pred_entity):
        para_index = paragraph.index(para)
        
        # retrieving labels from list
        for l in label:
            
            # count occurrence of each term
            gold_counter = Counter(gold_ent[l])
            pred_counter = Counter(pred_ent[l])
            
            # storing entities (label-wise) for document-level calculation
            all_gold_ent[l].update(gold_counter)  # CHECK: IF SAME TERM COMES FROM 2ND PARAGRAPH
            all_pred_ent[l].update(pred_counter)
            
            # calculate confusion matrix
            tp_counter = gold_counter & pred_counter  # Intersection of counts
            tp_sum = sum(tp_counter.values())

            fp_counter = pred_counter - gold_counter  # Predicted but not in gold
            fp_sum = sum(fp_counter.values())

            fn_counter = gold_counter - pred_counter  # Gold but not in predicted
            fn_sum = sum(fn_counter.values())

            # calculate precision, recall and f1-score for each paragraph
            precision = tp_sum / (tp_sum + fp_sum) if pred_counter else 0
            recall = tp_sum / (tp_sum + fn_sum) if gold_counter else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            # storing performance for each paragraph in list
            eval_metric_para.extend([f'{para_index} {l.upper()} Precision', 
                                     f'{para_index} {l.upper()} Recall', 
                                     f'{para_index} {l.upper()} F1'])

            eval_value_para.extend([f'{precision:.2f}',
                                    f'{recall:.2f}', 
                                    f'{f1:.2f}'])
    
    # creating dataframe from list
    df_score_para['metric'] = eval_metric_para
    df_score_para['value'] = eval_value_para
    
    # downloading paragraph-level performace in a spreadsheet
    file_name = 'score-para-ALL-ENT.xlsx'
    download_path = set_download_path(dir_path, file_name)
    df_score_para.to_excel(download_path, index=False)
    
    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => root directory')

    # document-level performance calculation  
    # variable declaration
    df_score_doc = pd.DataFrame(columns=['metric', 'value'])
    eval_metric_doc = []
    eval_value_doc = []
    
    # retrieving label from list
    for l in label:

        # calculate confusion matrix
        tp_counter = all_gold_ent[l] & all_pred_ent[l]
        tp_sum = sum(tp_counter.values())

        fp_counter = all_pred_ent[l] - all_gold_ent[l]
        fp_sum = sum(fp_counter.values())

        fn_counter = all_gold_ent[l] - all_pred_ent[l]
        fn_sum = sum(fn_counter.values())
        
        # calculate precision, recall and f1-score for entire document
        precision = tp_sum / (tp_sum + fp_sum) if all_pred_ent[l] else 0
        recall = tp_sum / (tp_sum + fn_sum) if all_gold_ent[l] else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f'{l}:\t{f1:.2f}(f) | {precision:.2f}(p) | {recall:.2f}(r)')
        
        # storing performance for the entire document in list
        eval_metric_doc.extend([f'{l.upper()} (Overall) Precision', 
                                f'{l.upper()} (Overall) Recall', 
                                f'{l.upper()} (Overall) F1'])
        
        eval_value_doc.extend([f'{precision:.2f}',
                               f'{recall:.2f}', 
                               f'{f1:.2f}'])
    
    # creating dataframe from list
    df_score_doc['metric'] = eval_metric_doc
    df_score_doc['value'] = eval_value_doc
    
    # downloading document-level performace in a spreadsheet
    file_name = 'score-doc-ALL-ENT.xlsx'
    download_path = set_download_path(dir_path, file_name)
    df_score_doc.to_excel(download_path, index=False)
    
    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => root directory')
       

## *Main:* (for performance evaluation)

#### <span style="color: blue;">Description:</span>
- calls 
    - ***ner_zero_shot()*** 
    - ***evaluate_distinct_entities()***
    - ***evaluate_all_entities()***
- generates files through *performance_metrics()*
    - *dl-all-terms.txt*
    - *dl-performance-paragraph.xlsx*
    - *dl-performance-overall.xlsx*
- generates files *evaluation_variables.py*

#### <span style="color: red;">Check:</span>
- *file_path* for evaluation data
- *labels* in case of merging or changing labels
- argument value (to set temperature) in *annotator_without_examples()* calling
- *download_location* for saving files

In [13]:
'''
Script Description:
 - Reads NER evaluation data from a text file.
 - Separates paragraphs and gold standard terms.
 - Parses the gold terms from JSON strings into dictionaries.
 - Performs zero-shot NER prediction using GPT-4o for each paragraph.
 - Ensures that predictions include all expected labels.
 - Saves evaluation variables to a file for reproducibility.
 - Evaluates model performance using a custom evaluation function.
'''
import json

def process_input_data(text_file):
    # read input data from text file
    with open(text_file, 'r', encoding='utf-8') as file:
        list_ = file.read().splitlines()

    # store paragraphs and annotations in different lists
    paragraph = []
    gold_ent_str = []

    for item in list_:
        if list_.index(item) == 0 or list_.index(item) % 2 == 0:
            paragraph.append(item)
        else:
            gold_ent_str.append(item)

    # convert annotations (in string format) to nested object
    gold_entity = []

    for item in gold_ent_str:
        try:
            json_obj = json.loads(item)              # Convert to dictionary
            gold_entity.append(json_obj)  # Add to list of dictionaries
        except json.JSONDecodeError as e:
            print(f'Error decoding JSON for item: {item}\nError: {e}')

    return paragraph, gold_entity


def annotate(instruction, paragraph, label, gold_entity, model, temp, dir_path=None):
    # predict entities from each paragraph for every instruction
    pred_entity = []
    for para in paragraph:
        pred_ent_in_para = dict()

        for inst in instruction:
            response = ner_zero_shot(inst, para, model, temp)    # calling annotator()
            if response:
                llm_label = list(response.keys())
                llm_ent = list(response.values())
                pred_ent_in_para[llm_label[0]] = llm_ent[0]
    #         else:
    #             print("ERROR::", response)
        
        # check for missing labels in predicted entities
        # all labels should be there even if they do not have any entities 
        if len(pred_ent_in_para) < len(label):    
            print('Label missing in predicted data.')            
            revised_data = dict()
            
            for l in label:
                if l not in pred_ent_in_para:
                    pred_ent_in_para[l] = []
                    print(f'Label -- {l} -- added to predicted data.')
            
            # organize the annotations' labels according to label's order
            for l in label:
                revised_data[l] = pred_ent_in_para[l]
                
            pred_ent_in_para = revised_data
        
        pred_entity.append(pred_ent_in_para)
    
    # save label, paragraph, gold_entity, pred_entity variables
    file_name = 'evaluation_variable.py'
    download_path = set_download_path(dir_path, file_name)
    with open(download_path, 'w', encoding='utf-8') as file:
        file.write('label = ' + repr(label) + '\n')
        file.write('paragraph = ' + repr(paragraph) + '\n')
        file.write('gold_entity = ' + repr(gold_entity) + '\n')
        file.write('pred_entity = ' + repr(pred_entity) + '\n')
        
    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => root directory')
    
    return pred_entity


In [18]:
from chatgpt_prompt import instruction

label = [
    'chemical',
    'material',
    'structure',
    'property',
    'application',
    'process',
    'equipment',
    'measurement',
    'abbreviation'
]

paragraph, gold_ent = process_input_data(text_file='sample-eval-data.txt')

pred_ent = annotate(
    instruction=instruction,
    paragraph=paragraph,
    label=label,
    gold_entity=gold_ent,
    model='gpt-4o',
    temp=0.2
    dir_path='output/zero-shot'
)

evaluate_distinct_entity(
    label=label,
    paragraph=paragraph,
    gold_entity=gold_ent,
    pred_entity=pred_ent
    dir_path='output/zero-shot'
)

evaluate_all_entity(
    label=label,
    paragraph=paragraph,
    gold_entity=gold_ent,
    pred_entity=pred_ent
    dir_path='output/zero-shot'
)

Downloaded evaluation_variable.py => None
Updated paragraph-with-all-entity.txt => None
Updated paragraph-with-all-entity.txt => None
Updated paragraph-with-all-entity.txt => None
Downloaded score-para-DIS-ENT.xlsx => None
chemical:	0.42(f) | 0.29(p) | 0.78(r)
material:	0.34(f) | 0.25(p) | 0.56(r)
structure:	0.61(f) | 0.74(p) | 0.52(r)
property:	0.39(f) | 0.32(p) | 0.50(r)
application:	0.00(f) | 0.00(p) | 0.00(r)
process:	0.67(f) | 1.00(p) | 0.50(r)
equipment:	0.00(f) | 0.00(p) | 0.00(r)
measurement:	0.75(f) | 0.60(p) | 1.00(r)
abbreviation:	0.00(f) | 0.00(p) | 0.00(r)
Downloaded score-doc-DIS-ENT.xlsx => None
Downloaded score-para-ALL-ENT.xlsx => None
chemical:	0.60(f) | 0.47(p) | 0.82(r)
material:	0.23(f) | 0.16(p) | 0.45(r)
structure:	0.63(f) | 0.75(p) | 0.54(r)
property:	0.33(f) | 0.27(p) | 0.43(r)
application:	0.00(f) | 0.00(p) | 0.00(r)
process:	0.62(f) | 1.00(p) | 0.45(r)
equipment:	0.00(f) | 0.00(p) | 0.00(r)
measurement:	0.75(f) | 0.60(p) | 1.00(r)
abbreviation:	0.00(f) | 0.00

# NEs Annotation Pipeline

In [2]:
import pandas as pd

def validate_entity_span(entity_in_paragraph):
    columns = ['paragraph', 'start_index', 'end_index', 'label', 'llm_term', 'sliced_term']
    df = pd.DataFrame(entity_in_paragraph, columns=columns)
    df_sorted = df.sort_values(by='start_index')
    df_sorted.reset_index(drop=True, inplace=True)
    df_sorted['equal'] = df_sorted['llm_term'] == df_sorted['sliced_term']
    df_sorted['overlap'] = False

    for i in range(1, len(df_sorted)):
        if df_sorted.loc[i, 'start_index'] >= df_sorted.loc[i-1, 'start_index'] and \
            df_sorted.loc[i, 'start_index'] <= df_sorted.loc[i-1, 'end_index']:
            df_sorted.loc[i, 'overlap'] = True

    df_sorted = df_sorted[df_sorted['equal']]     # Keep only where equal is True
    df_sorted = df_sorted[~df_sorted['overlap']]  # Remove rows with overlap

    list_ = []
    for index, row in df_sorted.iterrows():
        list_.append([row['start_index'], row['end_index'], row['label']])

    validated_entity = {'entities': list_}

    return validated_entity

In [3]:
import os
import json

# read evaluation data from text file and store paragraphs and entities in different list
def annotate(instruction, text_file, label, model, temp):

    with open(text_file, 'r', encoding='utf-8') as file:
        paragraph = file.read().splitlines()

    # predict entities from each paragraph for every instruction
    para_with_ent = []
    for para in paragraph:
        para_index = paragraph.index(para)
        ent_in_para = dict()

        for inst in instruction:
            response = ner_zero_shot(inst, para, model, temp)    # calling annotator()
            if response:
                llm_label = list(response.keys())
                llm_ent = list(response.values())
                ent_in_para[llm_label[0]] = llm_ent[0]
    #         else:
    #             print("ERROR::", response)

        ent_detail_in_para = []

        for key in ent_in_para:
            end_index = 0

            for ent in ent_in_para[key]:
                ent_length = len(ent)
                start_index = para.find(ent, end_index)

                if start_index != -1:
                    end_index = start_index + ent_length
                    ent_detail = [
                        para_index,                  # Paragraph number within a file
                        start_index,                 # Starting position of an entity
                        end_index,                   # Ending position of an entity
                        key.upper(),                 # Label of an entity
                        ent,                         # Entity extracted by llm
                        para[start_index:end_index]  # Entity extracted using start and end position
                    ]

                    ent_detail_in_para.append(ent_detail)

        validated_ent = validate_entity_span(ent_detail_in_para)
        para_with_ent.append([para, validated_ent])

    spacy_annotation = {'classes': label, 'annotations': para_with_ent}
    
    return spacy_annotation

    
def download_annotation(dict_obj, dir_path=None):
    file_name = 'spacy_annotation.json'
    download_path = set_download_path(dir_path, file_name)
    with open(download_path, 'w', encoding='utf-8') as json_file:
        json.dump(dict_obj, json_file, indent=4)
        
    if dir_path:
        print(f'Downloaded {file_name} => {dir_path}')
    else:
        print(f'Downloaded {file_name} => root directory')


## *Main:* (for continuous annotation)

In [5]:
from chatgpt_prompt import instruction

label = [
    'CHEMICAL',
    'MATERIAL',
    'STRUCTURE',
    'PROPERTY',
    'APPLICATION',
    'PROCESS',
    'EQUIPMENT',
    'MEASUREMENT',
    'ABBREVIATION'
]

spacy_annotation = annotate(
    instruction=instruction,
    text_file='sample-text-data.txt',
    label=label,
    model='gpt-4o',
    temp=0.2
)

download_annotation(dict_obj=spacy_annotation, dir_path='output/zero-shot)
